# ============================================================
# STAI-X Challenge 2026
# Data Merge for Baseline Models
#
# Version 1: One universal model
# Version 2: Three category-specific models
#
# Author: Sihang Jiang
# ============================================================

In [33]:
# ============================================================
# 1. Import packages and set project paths
# ============================================================

import os
import pandas as pd
import numpy as np

# Because this notebook is inside STAT-X/notebooks/
# and train/, val/, sample_submission.csv are in STAT-X/
BASE = ".."

TRAIN_Y_PATH = os.path.join(BASE, "train", "dose_sys_train.csv")
TRAIN_X_PATH = os.path.join(BASE, "train", "covariates_train.csv")
VAL_X_PATH = os.path.join(BASE, "val", "covariates.csv")
SUB_PATH = os.path.join(BASE, "sample_submission.csv")

OUTPUT_DIR = os.path.join(BASE, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [34]:
# ============================================================
# 2. Read four CSV files
# ============================================================

train_y = pd.read_csv(TRAIN_Y_PATH)
train_x = pd.read_csv(TRAIN_X_PATH)
val_x = pd.read_csv(VAL_X_PATH)
sub = pd.read_csv(SUB_PATH)

print("train_y:", train_y.shape)
print("train_x:", train_x.shape)
print("val_x:", val_x.shape)
print("sub:", sub.shape)



train_y: (31416, 4)
train_x: (3927, 12)
val_x: (357, 12)
sub: (918, 5)


In [35]:
# ============================================================
# 5. Keep only scoring categories for Award A prediction
# ============================================================

SCORING_CATEGORIES = ["all_drugs", "all_opioids", "all_stimulants"]

train_y_scored = train_y[
    train_y["overdose_category"].isin(SCORING_CATEGORIES)
].copy()

print("Filtered train_y_scored shape:", train_y_scored.shape)
print(train_y_scored["overdose_category"].value_counts())

Filtered train_y_scored shape: (11781, 4)
overdose_category
all_drugs         3927
all_opioids       3927
all_stimulants    3927
Name: count, dtype: int64


In [36]:
# ============================================================
# 6. Version 1: Merge data for one universal model
# ============================================================
#
# train_y_scored has one row per:
#   period_id + jurisdiction + overdose_category
#
# train_x has one row per:
#   period_id + jurisdiction
#
# Therefore, this is a many-to-one merge.
# The covariates are copied to each overdose category row.

train_universal = train_y_scored.merge(
    train_x,
    on=["period_id", "jurisdiction"],
    how="left",
    validate="many_to_one"
)

# Validation data:
# sub has one row per required prediction:
#   period_id + jurisdiction + overdose_category
#
# val_x has one row per:
#   period_id + jurisdiction
#
# This is also a many-to-one merge.
val_universal = sub.merge(
    val_x,
    on=["period_id", "jurisdiction"],
    how="left",
    validate="many_to_one"
)

print("train_universal:", train_universal.shape)
print("val_universal:", val_universal.shape)

display(train_universal.head())
display(val_universal.head())

train_universal: (11781, 14)
val_universal: (918, 15)


,period_id,jurisdiction,overdose_category,rate_per_10000_ed_visits,unemployment_rate,labor_force,temp_avg_f,precip_in,gtrends_overdose,gtrends_fentanyl,gtrends_naloxone,gtrends_opioid,gtrends_methamphetamine,state_doh_release
0,jtUOZLP4,AK,all_drugs,14.93,4.6,364680.0,NaN,NaN,49.56,56.70,54.74,46.15,31.88,Alaska Department of Health Authorizes Expande...
1,jtUOZLP4,AK,all_opioids,5.82,4.6,364680.0,NaN,NaN,49.56,56.70,54.74,46.15,31.88,Alaska Department of Health Authorizes Expande...
2,jtUOZLP4,AK,all_stimulants,5.39,4.6,364680.0,NaN,NaN,49.56,56.70,54.74,46.15,31.88,Alaska Department of Health Authorizes Expande...
3,jtUOZLP4,AL,all_drugs,23.63,2.9,2377299.0,66.8,5.65,55.97,64.03,61.82,52.13,36.00,NaN
4,jtUOZLP4,AL,all_opioids,8.97,2.9,2377299.0,66.8,5.65,55.97,64.03,61.82,52.13,36.00,NaN


,row_id,period_id,jurisdiction,overdose_category,rate_per_10000_ed_visits,unemployment_rate,labor_force,temp_avg_f,precip_in,gtrends_overdose,gtrends_fentanyl,gtrends_naloxone,gtrends_opioid,gtrends_methamphetamine,state_doh_release
0,0,aL5zkp6g,AK,all_drugs,0.0,4.6,365105.0,55.2,NaN,46.41,55.59,52.78,41.56,27.96,Alaska Department of Health Releases Provision...
1,1,aL5zkp6g,AK,all_opioids,0.0,4.6,365105.0,55.2,NaN,46.41,55.59,52.78,41.56,27.96,Alaska Department of Health Releases Provision...
2,2,aL5zkp6g,AK,all_stimulants,0.0,4.6,365105.0,55.2,NaN,46.41,55.59,52.78,41.56,27.96,Alaska Department of Health Releases Provision...
3,3,aL5zkp6g,AL,all_drugs,0.0,2.8,2375655.0,82.1,3.13,52.42,62.78,59.62,46.94,31.58,Alabama Department of Public Health Announces ...
4,4,aL5zkp6g,AL,all_opioids,0.0,2.8,2375655.0,82.1,3.13,52.42,62.78,59.62,46.94,31.58,Alabama Department of Public Health Announces ...


In [37]:
# ============================================================
# 7. Check merge quality for Version 1
# ============================================================

train_merge_check = train_y_scored.merge(
    train_x[["period_id", "jurisdiction"]],
    on=["period_id", "jurisdiction"],
    how="left",
    indicator=True
)

val_merge_check = sub.merge(
    val_x[["period_id", "jurisdiction"]],
    on=["period_id", "jurisdiction"],
    how="left",
    indicator=True
)

print("Training merge check:")
print(train_merge_check["_merge"].value_counts())

print("\nValidation merge check:")
print(val_merge_check["_merge"].value_counts())

Training merge check:
_merge
both          11781
left_only         0
right_only        0
Name: count, dtype: int64

Validation merge check:
_merge
both          918
left_only       0
right_only      0
Name: count, dtype: int64


In [38]:
# ============================================================
# 9. Save Version 1 merged data
# ============================================================

train_universal_path = os.path.join(OUTPUT_DIR, "train_universal_merged.csv")
val_universal_path = os.path.join(OUTPUT_DIR, "val_universal_merged.csv")

train_universal.to_csv(train_universal_path, index=False)
val_universal.to_csv(val_universal_path, index=False)

print("Saved Version 1 merged data:")
print(train_universal_path)
print(val_universal_path)

Saved Version 1 merged data:
..\outputs\train_universal_merged.csv
..\outputs\val_universal_merged.csv


In [39]:
# ============================================================
# 10. Version 2: Create category-specific merged datasets
# ============================================================

train_by_category = {}
val_by_category = {}

for cat in SCORING_CATEGORIES:
    print("\n" + "=" * 60)
    print("Preparing data for category:", cat)
    
    # Training rows for this category only
    train_y_cat = train_y_scored[
        train_y_scored["overdose_category"] == cat
    ].copy()
    
    # Submission rows for this category only
    sub_cat = sub[
        sub["overdose_category"] == cat
    ].copy()
    
    # Merge category-specific training target with training covariates
    train_cat = train_y_cat.merge(
        train_x,
        on=["period_id", "jurisdiction"],
        how="left",
        validate="one_to_one"
    )
    
    # Merge category-specific submission rows with validation covariates
    val_cat = sub_cat.merge(
        val_x,
        on=["period_id", "jurisdiction"],
        how="left",
        validate="many_to_one"
    )
    
    train_by_category[cat] = train_cat
    val_by_category[cat] = val_cat
    
    print("train_cat:", train_cat.shape)
    print("val_cat:", val_cat.shape)
    
    display(train_cat.head())
    display(val_cat.head())


Preparing data for category: all_drugs
train_cat: (3927, 14)
val_cat: (306, 15)


,period_id,jurisdiction,overdose_category,rate_per_10000_ed_visits,unemployment_rate,labor_force,temp_avg_f,precip_in,gtrends_overdose,gtrends_fentanyl,gtrends_naloxone,gtrends_opioid,gtrends_methamphetamine,state_doh_release
0,jtUOZLP4,AK,all_drugs,14.93,4.6,364680.0,NaN,NaN,49.56,56.70,54.74,46.15,31.88,Alaska Department of Health Authorizes Expande...
1,jtUOZLP4,AL,all_drugs,23.63,2.9,2377299.0,66.8,5.65,55.97,64.03,61.82,52.13,36.00,NaN
2,jtUOZLP4,AR,all_drugs,20.89,3.9,1429050.0,64.3,9.97,58.30,66.70,64.40,54.30,37.50,Arkansas Department of Health Expands Access t...
3,jtUOZLP4,AZ,all_drugs,18.99,4.2,3792200.0,58.6,0.34,54.80,62.70,60.54,51.04,35.25,Arizona Department of Health Services Pilots D...
4,jtUOZLP4,CA,all_drugs,14.69,5.4,19792412.0,56.4,0.80,52.47,60.03,57.96,48.87,33.75,California Department of Public Health Issues ...


,row_id,period_id,jurisdiction,overdose_category,rate_per_10000_ed_visits,unemployment_rate,labor_force,temp_avg_f,precip_in,gtrends_overdose,gtrends_fentanyl,gtrends_naloxone,gtrends_opioid,gtrends_methamphetamine,state_doh_release
0,0,aL5zkp6g,AK,all_drugs,0.0,4.6,365105.0,55.2,NaN,46.41,55.59,52.78,41.56,27.96,Alaska Department of Health Releases Provision...
1,3,aL5zkp6g,AL,all_drugs,0.0,2.8,2375655.0,82.1,3.13,52.42,62.78,59.62,46.94,31.58,Alabama Department of Public Health Announces ...
2,6,aL5zkp6g,AR,all_drugs,0.0,4.1,1434815.0,82.7,2.88,54.60,65.40,62.10,48.90,32.90,Arkansas Department of Health Warns Public of ...
3,9,aL5zkp6g,AZ,all_drugs,0.0,4.3,3802368.0,81.5,0.83,51.32,61.48,58.37,45.97,30.93,NaN
4,12,aL5zkp6g,CA,all_drugs,0.0,5.5,19836296.0,75.2,0.17,49.14,58.86,55.89,44.01,29.61,California Department of Public Health Warns o...



Preparing data for category: all_opioids
train_cat: (3927, 14)
val_cat: (306, 15)


,period_id,jurisdiction,overdose_category,rate_per_10000_ed_visits,unemployment_rate,labor_force,temp_avg_f,precip_in,gtrends_overdose,gtrends_fentanyl,gtrends_naloxone,gtrends_opioid,gtrends_methamphetamine,state_doh_release
0,jtUOZLP4,AK,all_opioids,5.82,4.6,364680.0,NaN,NaN,49.56,56.70,54.74,46.15,31.88,Alaska Department of Health Authorizes Expande...
1,jtUOZLP4,AL,all_opioids,8.97,2.9,2377299.0,66.8,5.65,55.97,64.03,61.82,52.13,36.00,NaN
2,jtUOZLP4,AR,all_opioids,8.57,3.9,1429050.0,64.3,9.97,58.30,66.70,64.40,54.30,37.50,Arkansas Department of Health Expands Access t...
3,jtUOZLP4,AZ,all_opioids,6.93,4.2,3792200.0,58.6,0.34,54.80,62.70,60.54,51.04,35.25,Arizona Department of Health Services Pilots D...
4,jtUOZLP4,CA,all_opioids,6.05,5.4,19792412.0,56.4,0.80,52.47,60.03,57.96,48.87,33.75,California Department of Public Health Issues ...


,row_id,period_id,jurisdiction,overdose_category,rate_per_10000_ed_visits,unemployment_rate,labor_force,temp_avg_f,precip_in,gtrends_overdose,gtrends_fentanyl,gtrends_naloxone,gtrends_opioid,gtrends_methamphetamine,state_doh_release
0,1,aL5zkp6g,AK,all_opioids,0.0,4.6,365105.0,55.2,NaN,46.41,55.59,52.78,41.56,27.96,Alaska Department of Health Releases Provision...
1,4,aL5zkp6g,AL,all_opioids,0.0,2.8,2375655.0,82.1,3.13,52.42,62.78,59.62,46.94,31.58,Alabama Department of Public Health Announces ...
2,7,aL5zkp6g,AR,all_opioids,0.0,4.1,1434815.0,82.7,2.88,54.60,65.40,62.10,48.90,32.90,Arkansas Department of Health Warns Public of ...
3,10,aL5zkp6g,AZ,all_opioids,0.0,4.3,3802368.0,81.5,0.83,51.32,61.48,58.37,45.97,30.93,NaN
4,13,aL5zkp6g,CA,all_opioids,0.0,5.5,19836296.0,75.2,0.17,49.14,58.86,55.89,44.01,29.61,California Department of Public Health Warns o...



Preparing data for category: all_stimulants
train_cat: (3927, 14)
val_cat: (306, 15)


,period_id,jurisdiction,overdose_category,rate_per_10000_ed_visits,unemployment_rate,labor_force,temp_avg_f,precip_in,gtrends_overdose,gtrends_fentanyl,gtrends_naloxone,gtrends_opioid,gtrends_methamphetamine,state_doh_release
0,jtUOZLP4,AK,all_stimulants,5.39,4.6,364680.0,NaN,NaN,49.56,56.70,54.74,46.15,31.88,Alaska Department of Health Authorizes Expande...
1,jtUOZLP4,AL,all_stimulants,4.04,2.9,2377299.0,66.8,5.65,55.97,64.03,61.82,52.13,36.00,NaN
2,jtUOZLP4,AR,all_stimulants,5.67,3.9,1429050.0,64.3,9.97,58.30,66.70,64.40,54.30,37.50,Arkansas Department of Health Expands Access t...
3,jtUOZLP4,AZ,all_stimulants,4.18,4.2,3792200.0,58.6,0.34,54.80,62.70,60.54,51.04,35.25,Arizona Department of Health Services Pilots D...
4,jtUOZLP4,CA,all_stimulants,4.67,5.4,19792412.0,56.4,0.80,52.47,60.03,57.96,48.87,33.75,California Department of Public Health Issues ...


,row_id,period_id,jurisdiction,overdose_category,rate_per_10000_ed_visits,unemployment_rate,labor_force,temp_avg_f,precip_in,gtrends_overdose,gtrends_fentanyl,gtrends_naloxone,gtrends_opioid,gtrends_methamphetamine,state_doh_release
0,2,aL5zkp6g,AK,all_stimulants,0.0,4.6,365105.0,55.2,NaN,46.41,55.59,52.78,41.56,27.96,Alaska Department of Health Releases Provision...
1,5,aL5zkp6g,AL,all_stimulants,0.0,2.8,2375655.0,82.1,3.13,52.42,62.78,59.62,46.94,31.58,Alabama Department of Public Health Announces ...
2,8,aL5zkp6g,AR,all_stimulants,0.0,4.1,1434815.0,82.7,2.88,54.60,65.40,62.10,48.90,32.90,Arkansas Department of Health Warns Public of ...
3,11,aL5zkp6g,AZ,all_stimulants,0.0,4.3,3802368.0,81.5,0.83,51.32,61.48,58.37,45.97,30.93,NaN
4,14,aL5zkp6g,CA,all_stimulants,0.0,5.5,19836296.0,75.2,0.17,49.14,58.86,55.89,44.01,29.61,California Department of Public Health Warns o...


In [40]:
# ============================================================
# 11. Check merge quality for each category-specific dataset
# ============================================================

for cat in SCORING_CATEGORIES:
    print("\n" + "=" * 60)
    print("Merge check for category:", cat)
    
    train_y_cat = train_y_scored[
        train_y_scored["overdose_category"] == cat
    ].copy()
    
    sub_cat = sub[
        sub["overdose_category"] == cat
    ].copy()
    
    train_cat_check = train_y_cat.merge(
        train_x[["period_id", "jurisdiction"]],
        on=["period_id", "jurisdiction"],
        how="left",
        indicator=True
    )
    
    val_cat_check = sub_cat.merge(
        val_x[["period_id", "jurisdiction"]],
        on=["period_id", "jurisdiction"],
        how="left",
        indicator=True
    )
    
    print("Training merge:")
    print(train_cat_check["_merge"].value_counts())
    
    print("\nValidation merge:")
    print(val_cat_check["_merge"].value_counts())


Merge check for category: all_drugs
Training merge:
_merge
both          3927
left_only        0
right_only       0
Name: count, dtype: int64

Validation merge:
_merge
both          306
left_only       0
right_only      0
Name: count, dtype: int64

Merge check for category: all_opioids
Training merge:
_merge
both          3927
left_only        0
right_only       0
Name: count, dtype: int64

Validation merge:
_merge
both          306
left_only       0
right_only      0
Name: count, dtype: int64

Merge check for category: all_stimulants
Training merge:
_merge
both          3927
left_only        0
right_only       0
Name: count, dtype: int64

Validation merge:
_merge
both          306
left_only       0
right_only      0
Name: count, dtype: int64


In [41]:
# ============================================================
# 13. Save Version 2 category-specific merged data
# ============================================================

for cat in SCORING_CATEGORIES:
    train_cat_path = os.path.join(OUTPUT_DIR, f"train_{cat}_merged.csv")
    val_cat_path = os.path.join(OUTPUT_DIR, f"val_{cat}_merged.csv")
    
    train_by_category[cat].to_csv(train_cat_path, index=False)
    val_by_category[cat].to_csv(val_cat_path, index=False)
    
    print("Saved:", train_cat_path)
    print("Saved:", val_cat_path)

Saved: ..\outputs\train_all_drugs_merged.csv
Saved: ..\outputs\val_all_drugs_merged.csv
Saved: ..\outputs\train_all_opioids_merged.csv
Saved: ..\outputs\val_all_opioids_merged.csv
Saved: ..\outputs\train_all_stimulants_merged.csv
Saved: ..\outputs\val_all_stimulants_merged.csv


In [42]:
# ============================================================
# 14. Final shape check
# ============================================================

print("Version 1: Universal model data")
print("train_universal:", train_universal.shape)
print("val_universal:", val_universal.shape)

print("\nVersion 2: Category-specific model data")
for cat in SCORING_CATEGORIES:
    print(cat)
    print("  train:", train_by_category[cat].shape)
    print("  val:", val_by_category[cat].shape)

print("\nExpected validation total rows:")
print("sub:", sub.shape[0])
print("val_universal:", val_universal.shape[0])
print("sum val_by_category:", sum(val_by_category[cat].shape[0] for cat in SCORING_CATEGORIES))

Version 1: Universal model data
train_universal: (11781, 14)
val_universal: (918, 15)

Version 2: Category-specific model data
all_drugs
  train: (3927, 14)
  val: (306, 15)
all_opioids
  train: (3927, 14)
  val: (306, 15)
all_stimulants
  train: (3927, 14)
  val: (306, 15)

Expected validation total rows:
sub: 918
val_universal: 918
sum val_by_category: 918
